In [0]:
%sql
USE CATALOG ecommerce;
USE SCHEMA gold;

In [0]:
df_orders = spark.read.table("ecommerce.silver.orders")
df_customers = spark.read.table("ecommerce.silver.customers")
df_orders.display()
df_customers.display()

OrderID,CustomerID,FirstName,LastName,State,City,OrderDate,OrderSeason,OrderDayOfWeek,ProductCategory,ProductSubCategory,UnitPrice,Quantity,TotalAmount,DiscountPercent,DiscountAmount,FinalAmount,PaymentMethod,DeviceType,MembershipTier,ShippingDays,Returned,ReviewScore
ORD-010020,CUST-05931,Matthew,King,ND,Madison,2022-03-31,Spring,Thursday,Clothing,Footwear,76.65,1,76.65,0,0.0,76.65,Paypal,Mobile,Silver,5,No,5
ORD-010290,CUST-05706,Kenneth,Green,AL,Bristol,2024-01-16,Winter,Tuesday,Health & Wellness,Fitness Accessories,10.65,2,21.3,20,4.26,17.04,Gift Card,Mobile,Gold,1,No,4
ORD-010606,CUST-07939,Robert,Harris,VA,Salem,2024-12-19,Winter,Thursday,Clothing,Bottoms,182.69,1,182.69,5,9.13,173.56,Apple Pay,Tablet,Platinum,5,No,4
ORD-011038,CUST-02982,Amanda,Baker,DE,Madison,2024-01-26,Winter,Friday,Clothing,Footwear,77.94,1,77.94,0,0.0,77.94,Apple Pay,Mobile,Silver,5,No,3
ORD-011144,CUST-07598,William,Taylor,AL,Clinton,2024-03-17,Spring,Sunday,Books,Self-help,19.52,2,39.04,0,0.0,39.04,Debit Card,Tablet,Silver,2,No,4
ORD-011574,CUST-06054,Ashley,White,NY,Rochester,2024-12-26,Winter,Thursday,Sports & Outdoors,Fitness Equipment,412.73,3,1238.19,10,123.82,1114.37,Debit Card,Desktop,Bronze,3,Yes,3
ORD-011615,CUST-07683,Betty,Roberts,MN,Springfield,2024-01-01,Winter,Monday,Health & Wellness,Vitamins,130.78,1,130.78,10,13.08,117.7,Apple Pay,Tablet,Silver,6,No,1
ORD-011624,CUST-08372,Susan,Allen,LA,Clinton,2023-09-23,Fall,Saturday,Grocery,Organic,11.59,2,23.18,0,0.0,23.18,Apple Pay,Mobile,Gold,5,No,4
ORD-011716,CUST-03792,Sarah,Brown,HI,Salem,2023-10-16,Fall,Monday,Beauty & Personal Care,Fragrances,16.69,3,50.07,0,0.0,50.07,Gift Card,Tablet,Platinum,5,No,4
ORD-010121,CUST-01092,Jessica,King,NJ,Springfield,2023-08-26,Summer,Saturday,Electronics,Laptops,587.15,2,1174.3,0,0.0,1174.3,Apple Pay,Desktop,Bronze,3,No,2


customer_id,customer_name,city
CUST-01002,Amanda Mitchell,Bristol
CUST-01004,Kevin Brown,Fairview
CUST-01005,Lisa Hall,Bristol
CUST-01007,Joshua Baker,Chicago
CUST-01008,Carol Baker,Clinton
CUST-01012,Jason Wright,Greenville
CUST-01016,Kenneth Scott,Columbus
CUST-01026,Thomas Adams,Albany
CUST-01027,Jessica Carter,Greensboro
CUST-01032,Nancy Brown,Clinton


In [0]:
from pyspark.sql.functions import col
orders = df_orders.alias("o")
customers =df_customers.alias("c")

customer_orders_gold = orders.join(customers,
    col("o.CustomerID") == col("c.customer_id"),
    "inner"
).select( col("o.OrderID"),col("o.OrderDate"),col("c.customer_id"),col("c.customer_name"),col("c.city"),col("o.ProductCategory"), col("o.ProductSubCategory"),col("o.Quantity"),col("o.FinalAmount"), col("o.PaymentMethod"),col("o.DeviceType"),
col("o.MembershipTier"),col("o.ShippingDays"),col("o.Returned")
)

customer_orders_gold.display()

OrderID,OrderDate,customer_id,customer_name,city,ProductCategory,ProductSubCategory,Quantity,FinalAmount,PaymentMethod,DeviceType,MembershipTier,ShippingDays,Returned
ORD-010020,2022-03-31,CUST-05931,Matthew King,Madison,Clothing,Footwear,1,76.65,Paypal,Mobile,Silver,5,No
ORD-010290,2024-01-16,CUST-05706,Kenneth Green,Bristol,Health & Wellness,Fitness Accessories,2,17.04,Gift Card,Mobile,Gold,1,No
ORD-010606,2024-12-19,CUST-07939,Robert Harris,Salem,Clothing,Bottoms,1,173.56,Apple Pay,Tablet,Platinum,5,No
ORD-011038,2024-01-26,CUST-02982,Amanda Baker,Madison,Clothing,Footwear,1,77.94,Apple Pay,Mobile,Silver,5,No
ORD-011144,2024-03-17,CUST-07598,William Taylor,Clinton,Books,Self-help,2,39.04,Debit Card,Tablet,Silver,2,No
ORD-011574,2024-12-26,CUST-06054,Ashley White,Rochester,Sports & Outdoors,Fitness Equipment,3,1114.37,Debit Card,Desktop,Bronze,3,Yes
ORD-011615,2024-01-01,CUST-07683,Linda Garcia,Georgetown,Health & Wellness,Vitamins,1,117.7,Apple Pay,Tablet,Silver,6,No
ORD-011624,2023-09-23,CUST-08372,Susan Allen,Clinton,Grocery,Organic,2,23.18,Apple Pay,Mobile,Gold,5,No
ORD-011716,2023-10-16,CUST-03792,Sarah Brown,Salem,Beauty & Personal Care,Fragrances,3,50.07,Gift Card,Tablet,Platinum,5,No
ORD-010121,2023-08-26,CUST-01092,Mary Scott,Fairview,Electronics,Laptops,2,1174.3,Apple Pay,Desktop,Bronze,3,No


In [0]:
customer_orders_gold.write.mode("overwrite").saveAsTable("ecommerce.gold.customer_orders")

In [0]:
#Reading the table from gold 
customer_orders_gold = spark.read.table("ecommerce.gold.customer_orders")

## Gold Layer: Business Insights (KPI Analysis)

1: Revenue by Product Category

In [0]:
from pyspark.sql.functions import sum,round

df_category = customer_orders_gold.groupBy("ProductCategory") \
    .agg(
        round(sum("FinalAmount" ),2).alias("total_revenue"),
        sum("Quantity").alias("total_quantity")
    ) \
    .orderBy("total_revenue", ascending=False)

df_category.display()

ProductCategory,total_revenue,total_quantity
Electronics,176461.76,269
Sports & Outdoors,59169.29,254
Home & Kitchen,45971.81,237
Automotive,36379.9,256
Clothing,26563.18,268
Health & Wellness,18392.58,271
Toys & Games,18300.6,243
Beauty & Personal Care,10386.54,240
Grocery,7055.09,267
Books,6587.85,238


In [0]:
df_category.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_category")

2: Average Order Value 

Indicates the average amount spent per order, useful for pricing and sales strategy

In [0]:
from pyspark.sql.functions import avg, round

df_aov = customer_orders_gold.select(
    round(avg("FinalAmount"), 2).alias("avg_order_value")
)

df_aov.display()

avg_order_value
233.18


In [0]:
df_aov.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_aov")

3: Membership Tier Analysis

Highlights which membership tiers contribute most to revenue, enabling targeted customer strategies.

In [0]:
df_membership = customer_orders_gold.groupBy("MembershipTier") \
    .agg(round(sum("FinalAmount"),2).alias("total_revenue")) \
    .orderBy("total_revenue", ascending=False)

df_membership.display()

MembershipTier,total_revenue
Gold,106517.01
Silver,103735.18
Bronze,98395.1
Platinum,96621.31


In [0]:
df_membership.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_membership")

4: Top 10 Customers

High-value customer identification

In [0]:
df_customers = customer_orders_gold.groupBy("customer_name") \
    .agg(
        round(sum("FinalAmount"),2).alias("total_spent")
    ) \
    .orderBy("total_spent", ascending=False) \
    .limit(10)

df_customers.display()

customer_name,total_spent
Donald Thompson,6022.02
Sarah Sanchez,4964.0
Lisa Hall,4582.74
Mark Green,4469.01
Jessica Adams,3892.59
Sharon Wilson,3277.64
Kenneth Lee,3241.97
Steven Campbell,3217.76
Jason Jackson,3198.72
Lisa Harris,3109.62


In [0]:
df_customers.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_top_customers")

5: Revenue by City

Shows revenue distribution across cities, helping in regional marketing and expansion planning.

In [0]:
df_city = customer_orders_gold.groupBy("city") \
    .agg(
        round(sum("FinalAmount"),2).alias("total_revenue")
    ) \
    .orderBy("total_revenue", ascending=False)

df_city.display()

city,total_revenue
Clinton,49383.5
Bristol,45929.62
Salem,42882.11
Georgetown,42338.91
Madison,38412.73
Springfield,35748.55
Fairview,35734.68
Greenville,34325.53
Durham,5641.86
Allentown,5515.2


In [0]:
df_city.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_city")

6: Payment Method Usage -
Customer behavior insight

In [0]:
df_payment = customer_orders_gold.groupBy("PaymentMethod") \
    .count().orderBy("count", ascending=False)

df_payment.display()

PaymentMethod,count
Buy Now Pay Later,277
Gift Card,269
Apple Pay,259
Paypal,240
Credit Card,239
Debit Card,235
Google Pay,219


In [0]:
df_payment.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_payment")

7: Return Analysis-
Product quality & satisfaction

In [0]:
from pyspark.sql.functions import count

df_return_rate = customer_orders_gold.groupBy("Returned") \
    .agg(count("*").alias("total_orders"))

df_return_rate.display()

Returned,total_orders
No,1628
Yes,110


In [0]:
df_return_rate.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_return")

8: Monthly Revenue

Shows monthly revenue trends, helping identify seasonality and support demand forecasting.

In [0]:
from pyspark.sql.functions import month, date_format

customer_orders_gold = customer_orders_gold \
    .withColumn("OrderMonth", month("OrderDate")) \
    .withColumn("OrderMonthName", date_format("OrderDate", "MMMM"))

df_month = customer_orders_gold.groupBy("OrderMonth", "OrderMonthName") \
    .agg(round(sum("FinalAmount"),2).alias("total_revenue")) \
    .orderBy("OrderMonth")

df_month.display()

OrderMonth,OrderMonthName,total_revenue
1,January,47956.35
2,February,23730.53
3,March,23978.38
4,April,31769.63
5,May,32413.47
6,June,28002.62
7,July,36910.24
8,August,25339.56
9,September,33570.95
10,October,45858.82


In [0]:
df_month.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_month")

## Gold Layer: Business Sales Summary (Unified Reporting Table)

In [0]:
from pyspark.sql.functions import sum, avg, count, round, when, col, countDistinct, year, month

df = customer_orders_gold

# Convert Returned column (Yes/No) → numeric (1/0) for calculations
df = df.withColumn(
    "is_return_flag",
    when(col("Returned") == "Yes", 1).otherwise(0)
)

# Extract year and month from OrderDate 
df = df.withColumn("order_year", year("OrderDate")) \
       .withColumn("order_month", month("OrderDate"))


df_reporting = df.groupBy(
    "order_year",          
    "order_month",        
    "ProductCategory",     
    "city",               
    "MembershipTier"       
).agg(
    round(sum("FinalAmount"), 2).alias("total_revenue"),   # total sales
    count("OrderID").alias("total_orders"),               # number of orders
    round(avg("FinalAmount"), 2).alias("avg_order_value"),# avg spend per order
    sum("Quantity").alias("total_units_sold"),            # total items sold
    countDistinct("customer_id").alias("unique_customers"),# unique buyers
    round((sum("is_return_flag") / count("OrderID")) * 100, 2).alias("return_rate_pct")  # return %
)


df_reporting.display()

order_year,order_month,ProductCategory,city,MembershipTier,total_revenue,total_orders,avg_order_value,total_units_sold,unique_customers,return_rate_pct
2024,6,Health & Wellness,Salem,Gold,15.37,1,15.37,1,1,0.0
2024,12,Electronics,Bristol,Bronze,5994.2,1,5994.2,5,1,0.0
2024,7,Clothing,Greenville,Bronze,63.07,1,63.07,1,1,0.0
2023,1,Sports & Outdoors,Clinton,Platinum,40.85,1,40.85,1,1,100.0
2023,1,Grocery,Fairview,Silver,106.85,2,53.43,3,2,0.0
2022,5,Grocery,Bristol,Silver,78.92,1,78.92,2,1,0.0
2022,5,Automotive,Salem,Platinum,113.28,1,113.28,1,1,0.0
2025,2,Beauty & Personal Care,Bristol,Silver,29.71,1,29.71,1,1,0.0
2023,9,Grocery,Springfield,Gold,12.05,1,12.05,1,1,0.0
2024,12,Beauty & Personal Care,Springfield,Silver,11.63,1,11.63,1,1,0.0


In [0]:
df_reporting.write.mode("overwrite").saveAsTable("ecommerce.gold.reporting_table")